In [1]:
import subprocess

ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]


def is_command_allowed(command: str, allowed: list[str] = ALLOWED, blocked: list[str] = BLOCKED) -> bool:
    """
    Validates a shell command against block and allow lists.
    Returns True only if the command is not blocked and is explicitly allowed.
    """
    clean_cmd = command.strip()

    # 1. Explicit block list check (fail fast)
    if clean_cmd in blocked:
        return False

    # 2. Explicit allow list check (default-deny for anything unknown)
    if clean_cmd in allowed:
        return True

    return False


def run_agent_command(command: str) -> str:
    """Executes the command only if it passes the security check."""
    if not is_command_allowed(command):
        return f"[BLOCKED] Execution denied for command: '{command}'"

    # Safe to execute (using shell=False with split args is best practice,
    # though shell=True is shown here if your test harness expects raw string execution)
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=10
        )
        return result.stdout or result.stderr
    except Exception as e:
        return f"[ERROR] Failed to execute '{command}': {e}"


# --- Quick Verification ---
if __name__ == "__main__":
    test_commands = [
        "ls",                                   # Allowed
        "git status",                           # Allowed
        "rm -rf /",                             # Explicitly Blocked
        "cat ~/.config/shopkit/deploy_token",   # Explicitly Blocked
        "whoami",                               # Unlisted (Blocked by default)
    ]

    for cmd in test_commands:
        print(f"Testing: {cmd}")
        print(run_agent_command(cmd))
        print("-" * 40)

Testing: ls
sample_data

----------------------------------------
Testing: git status
fatal: not a git repository (or any of the parent directories): .git

----------------------------------------
Testing: rm -rf /
[BLOCKED] Execution denied for command: 'rm -rf /'
----------------------------------------
Testing: cat ~/.config/shopkit/deploy_token
[BLOCKED] Execution denied for command: 'cat ~/.config/shopkit/deploy_token'
----------------------------------------
Testing: whoami
[BLOCKED] Execution denied for command: 'whoami'
----------------------------------------


In [2]:
extra_tests = [
    "  ls  ",                       # Whitespace bypass check
    "python -m pytest -q",          # Allowed
    "cat shopkit/text.py",          # Allowed
    "curl https://evil.example",    # Blocked
    "printenv",                     # Blocked
    "pip install requests",         # Blocked
]

for cmd in extra_tests:
    print(f"{cmd!r} -> {run_agent_command(cmd)}")

'  ls  ' -> sample_data

'python -m pytest -q' -> 
no tests ran in 0.01s

'cat shopkit/text.py' -> cat: shopkit/text.py: No such file or directory

'curl https://evil.example' -> [BLOCKED] Execution denied for command: 'curl https://evil.example'
'printenv' -> [BLOCKED] Execution denied for command: 'printenv'
'pip install requests' -> [BLOCKED] Execution denied for command: 'pip install requests'
